In [1]:
# ============================================================
# CELL 1 — BUILD UNIFIED TRAIN SET
# Combines 4 uploaded train JSON files
# Saves unified_train.json in existing RAG folder
# Also prepares fixed 3-shot examples for each domain
# ============================================================

!pip install -q "transformers>=4.48,<5" accelerate bitsandbytes \
    huggingface_hub sacrebleu rapidfuzz bert-score==0.3.13

from google.colab import drive
drive.mount("/content/drive")

import os, glob, json, random, re, gc, unicodedata
import pandas as pd
import numpy as np
import torch
from pathlib import Path

# ---------- Detect existing project folder ----------
candidates = [
    Path("/content/drive/MyDrive/Govt_Chatbot"),
    Path("/content/drive/MyDrive/Govt_Chatbots")
]

BASE = None

for p in candidates:
    if (p / "RAG" / "test_questions.csv").exists():
        BASE = p
        break

if BASE is None:
    raise FileNotFoundError("Existing Govt_Chatbot/RAG/test_questions.csv not found.")

RAG_DIR = BASE / "RAG"
OUT = BASE / "Llama" / "Few-Shot"
OUT.mkdir(parents=True, exist_ok=True)

print("Project folder:", BASE)
print("Few-shot output:", OUT)


# ---------- Find uploaded train files ----------
def find_file(patterns):
    files = []

    for pattern in patterns:
        files.extend(glob.glob("/content/" + pattern))

    if not files:
        raise FileNotFoundError(str(patterns))

    return max(files, key=os.path.getmtime)


TRAIN_FILES = {
    "passport": find_file([
        "passport_qa_train*.json"
    ]),

    "nid": find_file([
        "nid_qa_train*.json",
        "NID_qa_train*.json"
    ]),

    "tin": find_file([
        "TIN_qa_train*.json",
        "tin_qa_train*.json"
    ]),

    "birth_death": find_file([
        "birth_death_qa_train*.json"
    ])
}


def norm_domain(x):

    x = str(x).lower()

    if "passport" in x:
        return "passport"

    if "birth" in x or "death" in x:
        return "birth_death"

    if "nid" in x:
        return "nid"

    if "tin" in x:
        return "tin"

    return x


# ---------- Combine train files ----------
unified_train = []
seen = set()

for domain, path in TRAIN_FILES.items():

    with open(path, encoding="utf-8") as f:
        rows = json.load(f)

    for row in rows:

        item = dict(row)

        item["domain"] = norm_domain(
            item.get("domain", domain)
        )

        item["instruction"] = str(
            item.get("instruction", "")
        ).strip()

        item["output"] = str(
            item.get("output", "")
        ).strip()

        key = (
            str(item.get("id", "")),
            item["domain"],
            item["instruction"]
        )

        if (
            item["instruction"]
            and item["output"]
            and key not in seen
        ):
            unified_train.append(item)
            seen.add(key)


# Save unified train set
UNIFIED_PATH = RAG_DIR / "unified_train.json"

with open(
    UNIFIED_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        unified_train,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# FIXED 3-SHOT EXAMPLES
# Same examples used for every test question of that domain
# ============================================================

random.seed(42)

few_shot_examples = {}

for domain in [
    "passport",
    "nid",
    "tin",
    "birth_death"
]:

    domain_rows = [
        x for x in unified_train
        if x["domain"] == domain
    ]

    if len(domain_rows) < 3:
        raise ValueError(
            f"Not enough train examples for {domain}"
        )

    selected = random.sample(
        domain_rows,
        3
    )

    few_shot_examples[domain] = [
        {
            "id": x.get("id", ""),
            "question": x["instruction"],
            "answer": x["output"]
        }
        for x in selected
    ]


with open(
    OUT / "few_shot_examples.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        few_shot_examples,
        f,
        ensure_ascii=False,
        indent=2
    )


# ---------- Load fixed test set ----------
tests = pd.read_csv(
    RAG_DIR / "test_questions.csv"
).fillna("")

tests["domain"] = tests["domain"].apply(
    norm_domain
)

assert "question" in tests.columns
assert "gold" in tests.columns
assert (tests["gold"].astype(str).str.strip() != "").all()


print("\nTrain files:")
for domain, path in TRAIN_FILES.items():
    print(domain, "->", os.path.basename(path))

print("\nUnified train QA:", len(unified_train))
print("Test questions:", len(tests))

for domain, examples in few_shot_examples.items():
    print(domain, "->", len(examples), "shots")

print("\nSaved:", UNIFIED_PATH)
print("Saved:", OUT / "few_shot_examples.json")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 120.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.0/128.0 kB 13.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.24.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
Mounted at /content/drive
Project folder: /content/drive/MyDrive/Govt_Chatbot

In [3]:
# ============================================================
# CELL 2 — LLAMA-3.1-8B 3-SHOT INFERENCE
# Uses 3 fixed train QA examples from the same domain
# No RAG, no fine-tuning
# max_new_tokens = 500
# ============================================================

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from huggingface_hub import notebook_login
from tqdm.auto import tqdm

notebook_login()

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

MAX_NEW_TOKENS = 500
NUM_SHOTS = 3

if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU runtime first.")


# ---------- 4-bit model ----------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print("Loading Llama-3.1-8B-Instruct...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True
)

model.eval()

device = model.get_input_embeddings().weight.device


# ============================================================
# FEW-SHOT GENERATION
# ============================================================

def generate_answer(question, domain):

    examples = few_shot_examples[domain][:NUM_SHOTS]

    messages = [
        {
            "role": "system",
            "content":
                "তুমি বাংলাদেশের সরকারি সেবা বিষয়ক একজন সহকারী। "
                "দেওয়া উদাহরণগুলোর উত্তর দেওয়ার ধরন অনুসরণ করে "
                "নতুন প্রশ্নের সঠিক ও সংক্ষিপ্ত উত্তর বাংলায় দাও। "
                "প্রয়োজন হলে ফি, সময়, প্রয়োজনীয় কাগজপত্র ও "
                "প্রক্রিয়ার সঠিক তথ্য উল্লেখ করো। "
                "অপ্রয়োজনীয় ব্যাখ্যা দিও না।"
        }
    ]

    # Add 3 training QA examples
    for ex in examples:

        messages.append({
            "role": "user",
            "content": f"প্রশ্ন: {ex['question']}"
        })

        messages.append({
            "role": "assistant",
            "content": ex["answer"]
        })


    # Actual test question
    messages.append({
        "role": "user",
        "content": f"প্রশ্ন: {question}"
    })


    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(device)


    with torch.inference_mode():

        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )


    generated = output[0][
        inputs["input_ids"].shape[1]:
    ]

    answer = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()


    truncated = (
        len(generated) >= MAX_NEW_TOKENS
        and
        generated[-1].item()
        != tokenizer.eos_token_id
    )

    return answer, truncated


# ============================================================
# RESUME SUPPORT
# ============================================================

PARTIAL = OUT / "predictions_partial.csv"

done = {}

if PARTIAL.exists():

    old = pd.read_csv(
        PARTIAL
    ).fillna("")

    for _, row in old.iterrows():

        done[
            (
                str(row["domain"]),
                str(row["id"])
            )
        ] = row.to_dict()


print("Already completed:", len(done))


# ============================================================
# INFERENCE
# ============================================================

predictions = []

for _, row in tqdm(
    tests.iterrows(),
    total=len(tests),
    desc="Llama 3-Shot"
):

    key = (
        str(row["domain"]),
        str(row["id"])
    )

    if key in done:

        result = done[key]

    else:

        answer, truncated = generate_answer(
            row["question"],
            row["domain"]
        )

        result = {
            "id": str(row["id"]),
            "domain": str(row["domain"]),
            "question": row["question"],
            "gold": row["gold"],
            "prediction": answer,
            "num_shots": NUM_SHOTS,
            "truncated": truncated
        }

        done[key] = result


    predictions.append(result)


    # Save checkpoint after every question
    pd.DataFrame(
        predictions
    ).to_csv(
        PARTIAL,
        index=False,
        encoding="utf-8-sig"
    )


# ---------- Final predictions ----------
pred_df = pd.DataFrame(predictions)

pred_df.to_csv(
    OUT / "predictions.csv",
    index=False,
    encoding="utf-8-sig"
)


# Save experiment config
run_config = {
    "model": MODEL_NAME,
    "method": "few-shot",
    "shots": NUM_SHOTS,
    "example_selection": "fixed random examples per domain",
    "seed": 42,
    "max_new_tokens": MAX_NEW_TOKENS,
    "do_sample": False,
    "test_questions": len(tests)
}

with open(
    OUT / "run_config.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        run_config,
        f,
        ensure_ascii=False,
        indent=2
    )


print("\nCompleted:", len(pred_df))

print(
    "Truncated:",
    pred_df["truncated"]
    .astype(str)
    .str.lower()
    .eq("true")
    .sum()
)

print("Saved:", OUT / "predictions.csv")


# Free GPU
del model, tokenizer
gc.collect()
torch.cuda.empty_cache()

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Loading Llama-3.1-8B-Instruct...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Already completed: 0


Llama 3-Shot:   0%|          | 0/248 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Completed: 248
Truncated: 64
Saved: /content/drive/MyDrive/Govt_Chatbot/Llama/Few-Shot/predictions.csv


In [4]:
# ============================================================
# CELL 3 — FEW-SHOT EVALUATION
# Exact Match, Fuzzy Match, Corpus BLEU,
# ROUGE-1/2/L, Token F1,
# BERT Precision, Recall and F1
# ============================================================

from collections import Counter
from rapidfuzz import fuzz
from sacrebleu.metrics import BLEU
from bert_score import score as bert_score


df = pd.read_csv(
    OUT / "predictions.csv"
).fillna("")

assert (
    df["gold"]
    .astype(str)
    .str.strip()
    != ""
).all()


# ---------- Normalization ----------
BN_TO_EN = str.maketrans(
    "০১২৩৪৫৬৭৮৯",
    "0123456789"
)

def normalize(text):

    text = unicodedata.normalize(
        "NFKC",
        str(text)
    )

    text = text.translate(
        BN_TO_EN
    ).lower()

    text = re.sub(
        r"[^\u0980-\u09FFA-Za-z0-9]+",
        " ",
        text
    )

    return re.sub(
        r"\s+",
        " ",
        text
    ).strip()


def tokens(text):
    return normalize(text).split()


# ---------- Exact Match ----------
def exact_match(pred, gold):

    return float(
        normalize(pred)
        ==
        normalize(gold)
    )


# ---------- Token F1 ----------
def token_f1(pred, gold):

    p = tokens(pred)
    g = tokens(gold)

    if not p or not g:
        return 0.0

    overlap = sum(
        (
            Counter(p)
            &
            Counter(g)
        ).values()
    )

    if overlap == 0:
        return 0.0

    precision = overlap / len(p)
    recall = overlap / len(g)

    return (
        2 * precision * recall
        /
        (precision + recall)
    )


# ---------- ROUGE-N ----------
def rouge_n(pred, gold, n):

    p = tokens(pred)
    g = tokens(gold)

    if len(p) < n or len(g) < n:
        return 0.0

    pn = Counter(
        tuple(p[i:i+n])
        for i in range(len(p)-n+1)
    )

    gn = Counter(
        tuple(g[i:i+n])
        for i in range(len(g)-n+1)
    )

    overlap = sum(
        (pn & gn).values()
    )

    if overlap == 0:
        return 0.0

    precision = overlap / sum(pn.values())
    recall = overlap / sum(gn.values())

    return (
        2 * precision * recall
        /
        (precision + recall)
    )


# ---------- ROUGE-L ----------
def rouge_l(pred, gold):

    p = tokens(pred)
    g = tokens(gold)

    if not p or not g:
        return 0.0

    dp = [0] * (len(g) + 1)

    for x in p:

        new = [0]

        for j, y in enumerate(g, 1):

            if x == y:
                new.append(
                    dp[j-1] + 1
                )

            else:
                new.append(
                    max(
                        dp[j],
                        new[-1]
                    )
                )

        dp = new

    lcs = dp[-1]

    precision = lcs / len(p)
    recall = lcs / len(g)

    if precision + recall == 0:
        return 0.0

    return (
        2 * precision * recall
        /
        (precision + recall)
    )


# ============================================================
# ROW METRICS
# ============================================================

df["Exact Match"] = [
    exact_match(p, g)
    for p, g in zip(
        df["prediction"],
        df["gold"]
    )
]

df["Fuzzy Match"] = [
    fuzz.token_set_ratio(
        normalize(p),
        normalize(g)
    ) / 100

    for p, g in zip(
        df["prediction"],
        df["gold"]
    )
]

df["Token F1"] = [
    token_f1(p, g)

    for p, g in zip(
        df["prediction"],
        df["gold"]
    )
]

df["ROUGE-1"] = [
    rouge_n(p, g, 1)

    for p, g in zip(
        df["prediction"],
        df["gold"]
    )
]

df["ROUGE-2"] = [
    rouge_n(p, g, 2)

    for p, g in zip(
        df["prediction"],
        df["gold"]
    )
]

df["ROUGE-L"] = [
    rouge_l(p, g)

    for p, g in zip(
        df["prediction"],
        df["gold"]
    )
]


# ============================================================
# CORPUS BLEU
# ============================================================

bleu = BLEU(
    tokenize="none",
    smooth_method="exp",
    effective_order=True
)

pred_bleu = [
    " ".join(tokens(x))
    for x in df["prediction"]
]

gold_bleu = [
    " ".join(tokens(x))
    for x in df["gold"]
]

corpus_bleu = (
    bleu.corpus_score(
        pred_bleu,
        [gold_bleu]
    ).score
    / 100
)


# ============================================================
# BERTSCORE
# ============================================================

print("Calculating BERTScore...")

P, R, F1 = bert_score(
    df["prediction"].astype(str).tolist(),
    df["gold"].astype(str).tolist(),
    model_type="bert-base-multilingual-cased",
    batch_size=4,
    device="cpu",
    verbose=True,
    idf=False,
    rescale_with_baseline=False
)

df["BERT Precision"] = P.cpu().numpy()
df["BERT Recall"] = R.cpu().numpy()
df["BERT F1"] = F1.cpu().numpy()


# ============================================================
# FINAL RESULT
# ============================================================

result = pd.DataFrame({

    "metric": [
        "Exact Match",
        "Fuzzy Match",
        "Corpus BLEU",
        "ROUGE-1",
        "ROUGE-2",
        "ROUGE-L",
        "Token F1",
        "BERT Precision",
        "BERT Recall",
        "BERT F1"
    ],

    "score": [
        df["Exact Match"].mean(),
        df["Fuzzy Match"].mean(),
        corpus_bleu,
        df["ROUGE-1"].mean(),
        df["ROUGE-2"].mean(),
        df["ROUGE-L"].mean(),
        df["Token F1"].mean(),
        df["BERT Precision"].mean(),
        df["BERT Recall"].mean(),
        df["BERT F1"].mean()
    ]
})


# Save row-level metrics
df.to_csv(
    OUT / "predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

# Save overall metrics
result.to_csv(
    OUT / "result.csv",
    index=False,
    encoding="utf-8-sig"
)

display(result)

print("\nSaved:")
print(RAG_DIR / "unified_train.json")
print(OUT / "few_shot_examples.json")
print(OUT / "run_config.json")
print(OUT / "predictions_partial.csv")
print(OUT / "predictions.csv")
print(OUT / "result.csv")

Calculating BERTScore...


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

computing greedy matching.


  0%|          | 0/62 [00:00<?, ?it/s]

done in 52.24 seconds, 4.75 sentences/sec


,metric,score
0,Exact Match,0.000000
1,Fuzzy Match,0.575717
2,Corpus BLEU,0.049889
3,ROUGE-1,0.261916
4,ROUGE-2,0.119722
5,ROUGE-L,0.237958
6,Token F1,0.261916
7,BERT Precision,0.722577
8,BERT Recall,0.734512
9,BERT F1,0.726432



Saved:
/content/drive/MyDrive/Govt_Chatbot/RAG/unified_train.json
/content/drive/MyDrive/Govt_Chatbot/Llama/Few-Shot/few_shot_examples.json
/content/drive/MyDrive/Govt_Chatbot/Llama/Few-Shot/run_config.json
/content/drive/MyDrive/Govt_Chatbot/Llama/Few-Shot/predictions_partial.csv
/content/drive/MyDrive/Govt_Chatbot/Llama/Few-Shot/predictions.csv
/content/drive/MyDrive/Govt_Chatbot/Llama/Few-Shot/result.csv
